# Průzkumná analýza dat 
## Načtení a základní přehled dat

Před zahájením samotné analýzy si načtu kompletní dataset telekomunikačních dat, abych ověřila strukturu tabulky, datové typy a klíčové identifikátory.

In [0]:
SELECT * FROM workspace.default.telco 
LIMIT 20;


customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.3,1840.75,No
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.7,151.65,Yes
9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes
1452-KIOVK,Male,0,No,Yes,22,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,No,Month-to-month,Yes,Credit card (automatic),89.1,1949.4,No
6713-OKOMC,Female,0,No,No,10,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,No,Mailed check,29.75,301.9,No
7892-POOKP,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,104.8,3046.05,Yes
6388-TABGU,Male,0,No,Yes,62,Yes,No,DSL,Yes,Yes,No,No,No,No,One year,No,Bank transfer (automatic),56.15,3487.95,No


## Základní metriky a kontrola duplicit
Ověřím celkový počet záznamů v datasetu a ujistím se, že každé ID je unikátní.

In [0]:
SELECT 
  COUNT(*) AS celkovy_pocet_zakazniku,
  COUNT(DISTINCT customerID) AS pocet_unikatnich_zakazniku
FROM 
  workspace.default.telco;

celkovy_pocet_zakazniku,pocet_unikatnich_zakazniku
7043,7043


_Výsledek ukazuje, že jsou data v pořádku, nikde se neduplikují._


## Analýza odchodu zákazníků
Podívám se, kolik zákazníků z celkového počtu odešlo a jaké je procentuální zastoupení.

In [0]:
SELECT 
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS procento_z_celku
FROM 
  workspace.default.telco
GROUP BY 
  Churn;

Churn,pocet_zakazniku,procento_z_celku
No,5174,73.46
Yes,1869,26.54


Databricks visualization. Run in Databricks to view.

_Tímto jsem zjistila, že odchodovost zákazníků má celkem dost vysoké číslo a přidala jsem jednoduchý koláčový graf._

![Koláčový graf](images/kolacovy_graf.png)

## Vliv typu smlouvy na odchod zákazníků
Analyzuji, jaký vliv má smluvní závazek na míru odchodovosti. Předpokládám, že nejrizikovější budou smlouvy, které jsou na měsíční bázi.

In [0]:
SELECT 
  Contract,
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY Contract), 2) AS procento_v_ramci_smlouvy
FROM 
  workspace.default.telco
GROUP BY 
  Contract, 
  Churn
ORDER BY 
  Contract, 
  Churn;

Contract,Churn,pocet_zakazniku,procento_v_ramci_smlouvy
Month-to-month,No,2220,57.29
Month-to-month,Yes,1655,42.71
One year,No,1307,88.73
One year,Yes,166,11.27
Two year,No,1647,97.17
Two year,Yes,48,2.83


Databricks visualization. Run in Databricks to view.

_V kódu nepočítám procento z úplného celku, ale samostatně pro každý typ smlouvy, abych viděla kolik procent lidí odešlo z určité skupiny. Čili, když se podíváme na sloupec s procenty, tak vidíme že:_  

_&#45; u měsíčních smluv odejde 42,71 % zákazníků_  
_&#45; u ročních smluv je to 11,27 % zákazníků_  
_&#45; u dvouletých smluv to je 2,83 % zákazníků._

_Tady jsem přidala i sloupcový graf._

![Graf odchodovosti](images/graf_odchodovost.png)

## Finanční analýza a loajálnost
Podívám se jestli zákazníci kteří odcházejí, platí v průměru vyšší měsíční poplatky a po jaké době od firmy odcházejí.


In [0]:
SELECT 
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(AVG(MonthlyCharges), 2) AS prumerna_mesicni_platba,
  ROUND(AVG(tenure), 1) AS prumerna_doba_u_firmy_v_mesicich
FROM 
  workspace.default.telco
GROUP BY 
  Churn;

Churn,pocet_zakazniku,prumerna_mesicni_platba,prumerna_doba_u_firmy_v_mesicich
No,5174,61.27,37.6
Yes,1869,74.44,18.0


_Zde lze říct, že cena hraje určitě roli v rozhodování k odchodu._
_Klienti, kteří odcházejí platí v průměru 74,44 a ti věrní jen 61,27. Čili to znamená, že odcházejí ti co mají vlastně služby dražší._
_Když se podíváme na dobu jakou u nás jsou, tak jde vidět že vydrží kolem roka a půl. Možná bychom měli tento termín pohlídat a zkusit je kontaktovat s nabídkou výhodnějších služeb?_

# Analýza využívaných služeb a technické podpory

Zkoumám jaké konkrétní internetové služby zákazníci využívají a jestli má na jejich odchod vliv služba technické podpory. Chci ověřit, jestli drahé služby bez dostatečné péče nevedou k nespokojenosti a tudíž odchodu.

In [0]:
SELECT 
  CONCAT(InternetService, ' (', TechSupport, ')') AS sluzba_a_podpora,
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY InternetService, TechSupport), 2) AS procento_v_ramci_kombinace
FROM 
  workspace.default.telco
GROUP BY 
  InternetService,
  TechSupport,
  Churn
ORDER BY 
  InternetService,
  TechSupport,
  Churn;

sluzba_a_podpora,Churn,pocet_zakazniku,procento_v_ramci_kombinace
DSL (No),No,898,72.24
DSL (No),Yes,345,27.76
DSL (Yes),No,1064,90.32
DSL (Yes),Yes,114,9.68
Fiber optic (No),No,1129,50.63
Fiber optic (No),Yes,1101,49.37
Fiber optic (Yes),No,670,77.37
Fiber optic (Yes),Yes,196,22.63
No (No internet service),No,1413,92.60
No (No internet service),Yes,113,7.40


Databricks visualization. Run in Databricks to view.

![Graf odchodovosti dle typu internetu a technické podpory](images/graf_odchodovost_s_tech_sup.png)

_Zde vidíme, že nejrizikovější skupinou jsou zákazníci s optickým internetem, kteří nemají aktivovanou technickou podporu. Podle tabulky v této skupině odchází 49,37 % klientů. Pokud však zákazníci s optikou technickou podporu mají, míra odchodu klesá na 22,63 %._

_U DSL internetu bez podpory odejde 27,76 % zákazníků a s podporou odejde 9,68 % zákazníků, což je podobné jako u té optiky._

_Čili z toho vyplývá doporučení, že optický internet je potřeba nabízet nejlépe rovnou s technickou podporou._

# Analýza věrnosti zákazníků 

Nyní si uděláme select podle doby, co jsou klienti u naší společnosti. Ověříme tak, jestli je kritické období na začátku smluv a nebo ztrácíme klienty, kteří jsou u nás delší dobu.

In [0]:
SELECT 
  CASE 
    WHEN tenure <= 6 THEN '01. 0-6 měsíců'
    WHEN tenure <= 12 THEN '02. 6-12 měsíců'
    WHEN tenure <= 24 THEN '03. 1-2 roky'
    ELSE '04. Více než 2 roky'
  END AS segment_vernosti,
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY 
    CASE 
      WHEN tenure <= 6 THEN '01. 0-6 měsíců'
      WHEN tenure <= 12 THEN '02. 6-12 měsíců'
      WHEN tenure <= 24 THEN '03. 1-2 roky'
      ELSE '04. Více než 2 roky'
    END
  ), 2) AS procento_v_ramci_segmentu
FROM 
  workspace.default.telco
GROUP BY 
  1, 
  Churn
ORDER BY 
  segment_vernosti, 
  Churn;

segment_vernosti,Churn,pocet_zakazniku,procento_v_ramci_segmentu
01. 0-6 měsíců,No,697,47.06
01. 0-6 měsíců,Yes,784,52.94
02. 6-12 měsíců,No,452,64.11
02. 6-12 měsíců,Yes,253,35.89
03. 1-2 roky,No,730,71.29
03. 1-2 roky,Yes,294,28.71
04. Více než 2 roky,No,3295,85.96
04. Více než 2 roky,Yes,538,14.04


Databricks visualization. Run in Databricks to view.

![Graf věrnosti zákazníků](images/graf_vernosti.png)

_Zde vidíme, že obdobím pro odchod zákazníka je hned prvních 6 měsíců od začátku smlouvy. Odchází nám 52, 94% nově získaných zákazníků. Pak lze z tabulky vyčíst i to, že postupem času se to lepší. Ti, co tu jsou pak déle než 2 roky, tak míra odchodů klesá na 14, 04%._

_Daří se nám získávat nové klienty, ale nedokážeme je udržet. Bude tedy klíčové se na to zaměřit._




# Analýza vlivu platebních metod na odchod zákazníků
Zkoumám, zda má způsob platby vliv na loajalitu zákazníků. Předpokládám, že zákazníci, kteří musí platit účty manuálně, odcházejí častěji než ti, kteří mají nastavenou automatickou platbu z bankovního účtu.

In [0]:
SELECT 
  PaymentMethod,
  Churn,
  COUNT(*) AS pocet_zakazniku,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(PARTITION BY PaymentMethod), 2) AS procento_v_ramci_metody
FROM 
  workspace.default.telco
GROUP BY 
  PaymentMethod, 
  Churn
ORDER BY 
  PaymentMethod, 
  Churn;

PaymentMethod,Churn,pocet_zakazniku,procento_v_ramci_metody
Bank transfer (automatic),No,1286,83.29
Bank transfer (automatic),Yes,258,16.71
Credit card (automatic),No,1290,84.76
Credit card (automatic),Yes,232,15.24
Electronic check,No,1294,54.71
Electronic check,Yes,1071,45.29
Mailed check,No,1304,80.89
Mailed check,Yes,308,19.11


Databricks visualization. Run in Databricks to view.

![Graf odchodovosti podle platebních metod](images/graf_platebni_metody.png)

_Tak tady jasně vyplývá, že i volba platební metody má vliv na odchodovost zákazníků. U těch jednorázových plateb je odchodovost 45,29% a naopak u inkasních plateb se to pohybuje mezi 15 až 17%._

_Měli bychom zákazníky motivovat k tomu, aby si nastavovali inkasní způsoby plateb._
